# Explainable Road Damage Detection using YOLOv8 and Grad-CAM

## Training Notebook

This notebook contains the complete workflow for training a custom YOLOv8 model for road damage detection. It includes dataset preparation, model training, evaluation, inference, and preparation for Grad-CAM explainability analysis.

**Project Status:** Training Pipeline

In [ ]:
# ============================================
# Install Required Libraries
# ============================================

!pip install -q ultralytics

print("✅ Ultralytics Installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.0 MB/s eta 0:00:00
✅ Ultralytics Installed


In [ ]:
# ============================================
# Import Libraries
# ============================================

from ultralytics import YOLO

import os
import yaml
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Libraries Imported Successfully


# Step 4: Dataset Preparation

A custom road damage dataset is required because the pretrained YOLOv8 model is trained on the COCO dataset, which does not contain road damage classes such as potholes, longitudinal cracks, or transverse cracks.

The dataset used for training is organized in the YOLOv8 format, containing separate training, validation, and testing splits along with a configuration file (`data.yaml`).

In [ ]:
# ============================================
# Check GPU Availability
# ============================================

import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU : Tesla T4


In [ ]:
!pip install -q kaggle ultralytics

In [ ]:
from google.colab import files
files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("✅ Kaggle configured successfully")

In [ ]:
!kaggle datasets list -s "rdd2022" | head -10



In [ ]:
!kaggle datasets list -s "road damage yolo" | head -20

In [ ]:
# ============================================
# Download Road Damage Dataset
# ============================================

!kaggle datasets download -d lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes

In [ ]:
# ============================================
# Verify Downloaded Dataset
# ============================================

import os

for file in os.listdir():
    if file.endswith(".zip"):
        print("ZIP File:", file)
        print("Size (MB):", round(os.path.getsize(file) / (1024*1024), 2))

In [ ]:
# ============================================
# Extract Dataset
# ============================================

import zipfile

zip_file = "road-damage-dataset-potholes-cracks-and-manholes.zip"
extract_path = "RoadDamageDataset"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully!")

In [ ]:
# ============================================
# Inspect Dataset Structure
# ============================================

from pathlib import Path

dataset_path = Path("RoadDamageDataset")

for path in sorted(dataset_path.rglob("*")):
    if path.is_dir():
        print(path)

In [ ]:
# ============================================
# Inspect Images Folder
# ============================================

from pathlib import Path

image_path = Path("RoadDamageDataset/data/images")

print("Number of images:", len(list(image_path.glob("*"))))

print("\nFirst 10 images:")

for img in list(image_path.glob("*"))[:10]:
    print(img.name)

In [ ]:
# ============================================
# Inspect YOLO Labels
# ============================================

label_path = Path("RoadDamageDataset/data/labels-YOLO")

labels = list(label_path.glob("*.txt"))

print("Number of label files:", len(labels))

print("\nExample Label File:")

with open(labels[0]) as f:
    print(f.read())

In [ ]:
# ============================================
# Split Dataset into Train / Validation / Test
# ============================================

import os
import shutil
import random
from pathlib import Path

random.seed(42)

base = Path("RoadDamageDataset/data")

images = sorted((base / "images").glob("*"))
labels = base / "labels-YOLO"

dataset = Path("dataset")

for split in ["train", "valid", "test"]:
    (dataset / split / "images").mkdir(parents=True, exist_ok=True)
    (dataset / split / "labels").mkdir(parents=True, exist_ok=True)

random.shuffle(images)

n = len(images)

train = images[:int(0.7*n)]
valid = images[int(0.7*n):int(0.85*n)]
test = images[int(0.85*n):]

splits = {
    "train": train,
    "valid": valid,
    "test": test
}

for split_name, split_images in splits.items():

    for img in split_images:

        shutil.copy(img, dataset/split_name/"images"/img.name)

        label = labels/(img.stem + ".txt")

        if label.exists():
            shutil.copy(label, dataset/split_name/"labels"/label.name)

print("✅ Dataset Split Completed")

In [ ]:
# ============================================
# Verify Dataset Split
# ============================================

from pathlib import Path

dataset = Path("dataset")

for split in ["train", "valid", "test"]:

    img_count = len(list((dataset/split/"images").glob("*")))
    lbl_count = len(list((dataset/split/"labels").glob("*.txt")))

    print(f"{split.upper():6} Images: {img_count}")
    print(f"       Labels: {lbl_count}")
    print("-"*35)

In [ ]:
# ============================================
# Create data.yaml for YOLOv8
# ============================================

import yaml

data_yaml = {
    "path": "/content/dataset",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": [
        "crack",
        "manhole",
        "pothole"
    ]
}

with open("dataset/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("✅ data.yaml created successfully!")

In [ ]:
# ============================================
# Verify data.yaml
# ============================================

with open("dataset/data.yaml") as f:
    print(f.read())

In [ ]:
# ============================================
# Check Unique Class IDs
# ============================================

from pathlib import Path

label_folder = Path("RoadDamageDataset/data/labels-YOLO")

class_ids = set()

for txt_file in label_folder.glob("*.txt"):
    with open(txt_file, "r") as f:
        for line in f:
            if line.strip():
                class_ids.add(int(line.split()[0]))

print("Unique Class IDs:", sorted(class_ids))

In [ ]:
# ============================================
# Train YOLOv8 Model
# ============================================

from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

# Train the model
results = model.train(
    data="dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="RoadDamage_Training",
    name="yolov8n_road_damage",
    pretrained=True,
    verbose=True
)

In [ ]:
import os

model_path = "/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt"

print("Exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Size (MB):", round(os.path.getsize(model_path)/(1024*1024), 2))

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt")

results = model.predict(
    source="/content/dataset/test/images",
    save=True,
    imgsz=640,
    conf=0.25
)

print("Predictions completed!")

In [ ]:
import glob

folders = glob.glob("/content/runs/detect/predict*")

print(folders)

In [ ]:
import shutil
import glob
import os

pred_folder = "/content/runs/detect/predict"   # we'll change this if needed

os.makedirs("/content/project_results/predictions", exist_ok=True)

for img in glob.glob(pred_folder + "/*.jpg"):
    shutil.copy(img, "/content/project_results/predictions")

print("Copied:", len(glob.glob("/content/project_results/predictions/*.jpg")))

In [ ]:
import shutil

shutil.make_archive(
    "/content/road_damage_project_results",
    "zip",
    "/content/project_results"
)

print("ZIP created successfully!")

In [ ]:
import os

prediction_folder = "/content/runs/detect/predict"

print("Number of predicted images:",
      len(os.listdir(prediction_folder)))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

prediction_folder = Path("/content/runs/detect/predict-2")

images = sorted(prediction_folder.glob("*.jpg"))

for img_path in images[:5]:
    plt.figure(figsize=(8,6))
    plt.imshow(Image.open(img_path))
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
# ============================================
# Evaluate the Trained YOLOv8 Model
# ============================================

from ultralytics import YOLO

model = YOLO("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt")

metrics = model.val()

print(metrics)

In [ ]:
# ============================================
# Show Training Result Files
# ============================================

from pathlib import Path

results_dir = Path("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage")

print("Files generated during training:\n")

for file in sorted(results_dir.iterdir()):
    print(file.name)

In [ ]:
# ============================================
# Display Training Results
# ============================================

from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/results.png")

plt.figure(figsize=(16,10))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
# ============================================
# Display Confusion Matrix
# ============================================

from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/confusion_matrix.png")

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
# ============================================
# Display Precision-Recall Curve
# ============================================

from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/BoxPR_curve.png")

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
!pip install grad-cam

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt")

print(model.model)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt")

print(len(model.model.model))

In [ ]:
for i, layer in enumerate(model.model.model):
    print(i, ":", layer.__class__.__name__)

In [ ]:
from google.colab import files
import shutil

# Zip the training results
shutil.make_archive(
    "/content/RoadDamage_Project_Results",
    "zip",
    "/content/runs/detect/RoadDamage_Training/yolov8n_road_damage"
)

print("✅ ZIP created!")

In [ ]:
files.download("/content/RoadDamage_Project_Results.zip")

In [ ]:
import os

folders = [
    "/content/project_results",
    "/content/project_results/figures",
    "/content/project_results/predictions"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folders created.")

In [ ]:
import shutil

source = "/content/runs/detect/RoadDamage_Training/yolov8n_road_damage"

files = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png"
]

for f in files:
    shutil.copy(
        f"{source}/{f}",
        f"/content/project_results/figures/{f}"
    )

print("Figures copied successfully.")

In [ ]:
import shutil
import glob

pred_folder = "/content/runs/detect/predict-2"

images = glob.glob(pred_folder + "/*.jpg")

print("Predicted Images:", len(images))

for img in images[:10]:
    shutil.copy(img, "/content/project_results/predictions/")

In [ ]:
import shutil

shutil.make_archive(
    "/content/road_damage_project_results",
    "zip",
    "/content/project_results"
)

print("ZIP created.")

In [ ]:
import os

pred_folder = "/content/runs/detect/predict-2"

print("Exists:", os.path.exists(pred_folder))

if os.path.exists(pred_folder):
    files = os.listdir(pred_folder)
    print("Number of files:", len(files))
    print(files[:10])

In [ ]:
import os

print(os.listdir("/content/project_results"))
print()

print("Figures:")
print(os.listdir("/content/project_results/figures"))

print()

print("Predictions:")
print(os.listdir("/content/project_results/predictions"))

In [ ]:
!pip install ultralytics grad-cam -q

In [ ]:

from ultralytics import YOLO

model = YOLO("/content/runs/detect/RoadDamage_Training/yolov8n_road_damage/weights/best.pt")

for i, layer in enumerate(model.model.model):
    print(i, ":", layer.__class__.__name__)

In [ ]:
target_layer = model.model.model[21]

print(target_layer)